In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("cbb_data.csv")

df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")

# Keep favorites only (negative spread)
fav = df[df["CLOSING SPREAD"] < 0].copy()

# One row per game
games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           DATE=("DATE", "first")
       )
)

print("Total favorite games:", len(games))
games.head()

Total favorite games: 5760


,GAME-ID,SPREAD,DATE
0,401700174,-1.0,1/2/2025
1,401700175,-2.5,1/2/2025
2,401700176,-10.0,1/2/2025
3,401700177,-4.0,1/2/2025
4,401700178,-2.5,1/4/2025


In [3]:
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -9, -5, -1],
    labels=["-9 and above", "-5 to -8.5", "-1 to -4.5"],
    right=True
)

print(games["spread_bucket"].value_counts())

spread_bucket
-1 to -4.5      2251
-9 and above    1934
-5 to -8.5      1575
Name: count, dtype: int64


In [4]:
spread_counts = games["spread_bucket"].value_counts().sort_index()
print("\nTotal Games Per Spread Bucket:")
print(spread_counts)


Total Games Per Spread Bucket:
spread_bucket
-9 and above    1934
-5 to -8.5      1575
-1 to -4.5      2251
Name: count, dtype: int64


Now doing Total favourite wins 

In [5]:
import numpy as np
import pandas as pd

# Ensure numeric
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")

# Favorites only (one row per game, anchored on favorite)
fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           DATE=("DATE", "first")
       )
)

# Your new spread buckets
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -9, -5, -1],
    labels=["-9 and above", "-5 to -8.5", "-1 to -4.5"],
    right=True
)

# Total games per bucket
total_games_bucket = games.groupby("spread_bucket").size()

# Favorite wins per bucket
fav_wins_bucket = games[games["FAV_WIN"] == 1].groupby("spread_bucket").size()

# Combine into one table + win rate
summary = pd.DataFrame({
    "total_games": total_games_bucket,
    "fav_wins": fav_wins_bucket
}).fillna(0).astype(int)

summary["fav_win_rate"] = (summary["fav_wins"] / summary["total_games"]).round(4)

print(summary)

               total_games  fav_wins  fav_win_rate
spread_bucket                                     
-9 and above          1934      1774        0.9173
-5 to -8.5            1575      1155        0.7333
-1 to -4.5            2251      1303        0.5789


In [6]:

# Ensure numeric
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# Favorite rows only
fav = df[df["CLOSING SPREAD"] < 0].copy()

# One row per game (favorite side)
games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_WIN_OPP_OVER=("Team ML + Opp Team Over Win/Loss (1/0)", "first")
       )
)

# Spread buckets
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -9, -5, -1],
    labels=["-9 and above", "-5 to -8.5", "-1 to -4.5"],
    right=True
)

# Count of games where Favorite wins + Opp Over
joint_counts = (
    games[games["FAV_WIN_OPP_OVER"] == 1]
    .groupby("spread_bucket")
    .size()
)

print("Fav Win + Opp Over counts:")
print(joint_counts)

Fav Win + Opp Over counts:
spread_bucket
-9 and above    805
-5 to -8.5      427
-1 to -4.5      467
dtype: int64


In [7]:
# Ensure numeric
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# Favorites only
fav = df[df["CLOSING SPREAD"] < 0].copy()

# One row per game
games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_WIN_OPP_OVER=("Team ML + Opp Team Over Win/Loss (1/0)", "first")
       )
)

# Spread buckets
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -9, -5, -1],
    labels=["-9 and above", "-5 to -8.5", "-1 to -4.5"],
    right=True
)

# Denominator: Favorite wins
fav_wins = (
    games[games["FAV_WIN"] == 1]
    .groupby("spread_bucket")
    .size()
)

# Numerator: Favorite win + Opp Over
joint = (
    games[games["FAV_WIN_OPP_OVER"] == 1]
    .groupby("spread_bucket")
    .size()
)

# Conditional probability
prob = (joint / fav_wins).round(4)

result = pd.DataFrame({
    "Fav Wins": fav_wins,
    "Fav Win + Opp Over": joint,
    "P(Fav Win + Opp Over | Fav Win)": prob
})

print(result)

               Fav Wins  Fav Win + Opp Over  P(Fav Win + Opp Over | Fav Win)
spread_bucket                                                               
-9 and above       1774                 805                           0.4538
-5 to -8.5         1155                 427                           0.3697
-1 to -4.5         1303                 467                           0.3584


In [8]:
# Add Fair American Price column
result["Fair American Price"] = prob.apply(
    lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p
)

print(result)

               Fav Wins  Fav Win + Opp Over  P(Fav Win + Opp Over | Fav Win)  \
spread_bucket                                                                  
-9 and above       1774                 805                           0.4538   
-5 to -8.5         1155                 427                           0.3697   
-1 to -4.5         1303                 467                           0.3584   

               Fair American Price  
spread_bucket                       
-9 and above            120.372671  
-5 to -8.5              170.491803  
-1 to -4.5              179.014989  


In [9]:
# Fair American (no adjustment)
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

# 10% anticorrelation adjustment
boost = 0.10
boosted_prob = prob * (1 - boost)
boosted_price = boosted_prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

price_table = pd.DataFrame({
    "Fair American Price": fair_price.round(0),
    "Fair American Price (anticorrelation boost)": boosted_price.round(0)
})

print(price_table)

               Fair American Price  \
spread_bucket                        
-9 and above                 120.0   
-5 to -8.5                   170.0   
-1 to -4.5                   179.0   

               Fair American Price (anticorrelation boost)  
spread_bucket                                               
-9 and above                                         145.0  
-5 to -8.5                                           201.0  
-1 to -4.5                                           210.0  


Now doing it for 2 


In [10]:
import numpy as np
import pandas as pd

df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

fav = df[df["CLOSING SPREAD"] < 0].copy()

games = (
    fav.groupby("GAME-ID", as_index=False)
       .agg(
           SPREAD=("CLOSING SPREAD", "first"),
           FAV_WIN=("Team W/L (1/0)", "first"),
           FAV_WIN_OPP_OVER=("Team ML + Opp Team Over Win/Loss (1/0)", "first")
       )
)

games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -12, -8, -4, -1],
    labels=["-12 and above", "-8 to -11.5", "-4 to -7.5", "-1 to -3.5"],
    right=True
)

# ✅ Total games per bucket
total_games = games.groupby("spread_bucket").size()

# Denominator: favorite wins
fav_wins = games[games["FAV_WIN"] == 1].groupby("spread_bucket").size()

# Numerator: favorite win + opp over
joint = games[games["FAV_WIN_OPP_OVER"] == 1].groupby("spread_bucket").size()

prob = (joint / fav_wins)
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

result = pd.DataFrame({
    "Total Games": total_games,
    "Fav Wins": fav_wins,
    "Fav Win + Opp Over": joint,
    "P(Fav Win + Opp Over | Fav Win)": prob.round(4),
    "Fair American Price": fair_price.round(0)
}).fillna(0)

# make counts int
for c in ["Total Games", "Fav Wins", "Fav Win + Opp Over"]:
    result[c] = result[c].astype(int)

print(result)

               Total Games  Fav Wins  Fav Win + Opp Over  \
spread_bucket                                              
-12 and above         1276      1214                 564   
-8 to -11.5            973       809                 349   
-4 to -7.5            1749      1210                 439   
-1 to -3.5            1762       999                 347   

               P(Fav Win + Opp Over | Fav Win)  Fair American Price  
spread_bucket                                                        
-12 and above                           0.4646                115.0  
-8 to -11.5                             0.4314                132.0  
-4 to -7.5                              0.3628                176.0  
-1 to -3.5                              0.3473                188.0  


In [12]:
# --- New 4 spread buckets ---
games["spread_bucket"] = pd.cut(
    games["SPREAD"],
    bins=[-np.inf, -12, -8, -4, -1],
    labels=["-12 and above", "-8 to -11.5", "-4 to -7.5", "-1 to -3.5"],
    right=True
)

# Recompute probability: P(Fav Win + Opp Over | Fav Win)
fav_wins = games[games["FAV_WIN"] == 1].groupby("spread_bucket").size()
joint = games[games["FAV_WIN_OPP_OVER"] == 1].groupby("spread_bucket").size()
prob = joint / fav_wins

# Fair American (no adjustment)
fair_price = prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

# 10% anticorrelation adjustment
boost = 0.10
boosted_prob = prob * (1 - boost)
boosted_price = boosted_prob.apply(lambda p: -100*p/(1-p) if p > 0.5 else 100*(1-p)/p)

price_table = pd.DataFrame({
    "P(Fav Win + Opp Over | Fav Win)": prob.round(4),
    "Fair American Price": fair_price.round(0),
    "Fair American Price (anticorrelation boost)": boosted_price.round(0)
}).fillna(0)

print(price_table)

               P(Fav Win + Opp Over | Fav Win)  Fair American Price  \
spread_bucket                                                         
-12 and above                           0.4646                115.0   
-8 to -11.5                             0.4314                132.0   
-4 to -7.5                              0.3628                176.0   
-1 to -3.5                              0.3473                188.0   

               Fair American Price (anticorrelation boost)  
spread_bucket                                               
-12 and above                                        139.0  
-8 to -11.5                                          158.0  
-4 to -7.5                                           206.0  
-1 to -3.5                                           220.0  
